In [ ]:
import math
from dataclasses import dataclass, field
from typing import Optional, Tuple, List

EPS = 1e-9

# ---------------------------------------------------------
# 1. CORE GEOMETRY (Extended with Horocycles)
# ---------------------------------------------------------

@dataclass(frozen=True)
class HyperbolicPoint:
    """
    Represents a point in the (closed) upper half-plane model:
      - kind == "interior": true hyperbolic point (x,y) with y>0
      - kind == "boundary": ideal boundary point on R (y=0)
      - kind == "infinity": the point at ∞

    For "interior": we store x,y.
    For "boundary": x, y=0.
    For "infinity": x=y=None.
    """
    kind: str
    x: Optional[float] = None
    y: Optional[float] = None

    @staticmethod
    def interior(z: complex) -> "HyperbolicPoint":
        x = float(z.real)
        y = float(z.imag)
        if y <= 0:
            raise ValueError("Interior point must have Im(z) > 0.")
        return HyperbolicPoint("interior", x, y)

    @staticmethod
    def boundary(x: float) -> "HyperbolicPoint":
        return HyperbolicPoint("boundary", float(x), 0.0)

    @staticmethod
    def infinity() -> "HyperbolicPoint":
        return HyperbolicPoint("infinity", None, None)

    def as_xy(self) -> Tuple[Optional[float], Optional[float]]:
        if self.kind == "infinity":
            return (None, None)
        return (self.x, self.y if self.y is not None else 0.0)


@dataclass
class Horocycle:
    """
    A horocycle in the upper half-plane model.
    
    Two types:
    - "horizontal": y = h (horizontal line at height h > 0)
    - "circle": circle tangent to real axis at (c, 0) with radius R
    
    For horizontal: stores h
    For circle: stores (c, R)
    """
    kind: str  # "horizontal" or "circle"
    h: Optional[float] = None  # for horizontal
    c: Optional[float] = None  # for circle (center x-coord)
    R: Optional[float] = None  # for circle (radius)
    
    @staticmethod
    def horizontal(h: float) -> "Horocycle":
        """Create horizontal horocycle at height h."""
        if h <= 0:
            raise ValueError("Horizontal horocycle must have h > 0")
        return Horocycle(kind="horizontal", h=h)
    
    @staticmethod
    def circle_tangent(c: float, R: float) -> "Horocycle":
        """Create circular horocycle tangent to real axis at (c, 0) with radius R."""
        if R <= 0:
            raise ValueError("Circle horocycle must have R > 0")
        return Horocycle(kind="circle", c=c, R=R)
    
    def svg_element_uhp(
        self,
        transform,
        xmin: float,
        xmax: float,
        stroke: str = "blue",
        stroke_width: float = 1.5,
        stroke_dasharray: str = "5,3",
    ) -> str:
        """
        Return SVG element for this horocycle in UHP coordinates.
        """
        if self.kind == "horizontal":
            # Draw horizontal line from xmin to xmax at y = h
            x1_svg, y1_svg = transform.world_to_svg(xmin, self.h)
            x2_svg, y2_svg = transform.world_to_svg(xmax, self.h)
            return (
                f'<line x1="{x1_svg:.3f}" y1="{y1_svg:.3f}" '
                f'x2="{x2_svg:.3f}" y2="{y2_svg:.3f}" '
                f'stroke="{stroke}" stroke-width="{stroke_width}" '
                f'stroke-dasharray="{stroke_dasharray}"/>'
            )
        
        elif self.kind == "circle":
            # Draw circle with center (c, R) and radius R
            # This is a circle tangent to y=0 at point (c, 0)
            cx_svg, cy_svg = transform.world_to_svg(self.c, self.R)
            R_svg = transform.radius_to_svg(self.R)
            return (
                f'<circle cx="{cx_svg:.3f}" cy="{cy_svg:.3f}" r="{R_svg:.3f}" '
                f'stroke="{stroke}" stroke-width="{stroke_width}" '
                f'stroke-dasharray="{stroke_dasharray}" fill="none"/>'
            )
        
        else:
            raise ValueError(f"Unknown horocycle kind: {self.kind}")


@dataclass
class Geodesic:
    """
    A hyperbolic geodesic segment determined by two HyperbolicPoint endpoints
    (p1, p2) in the UHP model.

    Internally it is either:
      - "vertical": x = constant
      - "circle" : (x - c)^2 + y^2 = R^2 with center (c,0), R>0

    We assume p1 != p2, unless both are infinity (which is invalid).
    """
    p1: HyperbolicPoint
    p2: HyperbolicPoint

    def _circle_params(self):
        """
        Returns:
          ("vertical", x0)
        or
          ("circle", c, R)
        describing the full UHP geodesic that contains p1 and p2.
        """
        # exclude (∞,∞)
        if self.p1.kind == "infinity" and self.p2.kind == "infinity":
            raise ValueError("Geodesic not defined by (∞, ∞).")

        x1, y1 = self.p1.as_xy()
        x2, y2 = self.p2.as_xy()

        # If one is ∞ and other is finite -> vertical through that x
        if self.p1.kind == "infinity" and self.p2.kind != "infinity":
            return ("vertical", float(x2))
        if self.p2.kind == "infinity" and self.p1.kind != "infinity":
            return ("vertical", float(x1))

        # both are finite here
        assert x1 is not None and x2 is not None
        assert y1 is not None and y2 is not None

        # distinct?
        if abs(x1 - x2) < EPS and abs(y1 - y2) < EPS:
            raise ValueError("Need two distinct points for a geodesic segment.")

        # same x -> vertical
        if abs(x1 - x2) < EPS:
            return ("vertical", float(x1))

        # otherwise circle orthogonal to real axis:
        # center c on real axis, radius R
        # c = [ (x2^2 + y2^2) - (x1^2 + y1^2) ] / (2 (x2 - x1))
        num = (x2**2 + y2**2) - (x1**2 + y1**2)
        den = 2.0 * (x2 - x1)
        c = num / den
        R = math.hypot(x1 - c, y1)

        return ("circle", c, R)

    def svg_element(
        self,
        transform,
        stroke: str = "black",
        stroke_width: float = 2.0,
        ymax_clip: Optional[float] = None,
    ) -> str:
        """
        Return a standalone SVG element (<line> or <path>) for this geodesic
        segment in the UHP scene.

        transform: a Transform that maps (x,y)->(svgX,svgY), etc.
        ymax_clip: used to cap vertical rays to some finite y when ∞ is involved.
        """
        kind_params = self._circle_params()

        # vertical
        if kind_params[0] == "vertical":
            x0 = kind_params[1]

            def pick_y(p: HyperbolicPoint) -> float:
                if p.kind == "infinity":
                    # draw up to ymax_clip
                    if ymax_clip is None:
                        raise ValueError("Need ymax_clip for vertical segment to ∞.")
                    return ymax_clip
                # finite
                _, yy = p.as_xy()
                return yy

            y_start = pick_y(self.p1)
            y_end   = pick_y(self.p2)

            x1_svg, y1_svg = transform.world_to_svg(x0, y_start)
            x2_svg, y2_svg = transform.world_to_svg(x0, y_end)

            return (
                f'<line x1="{x1_svg:.3f}" y1="{y1_svg:.3f}" '
                f'x2="{x2_svg:.3f}" y2="{y2_svg:.3f}" '
                f'stroke="{stroke}" stroke-width="{stroke_width}" '
                f'fill="none"/>'
            )

        # circle arc
        _, c, R = kind_params

        x1, y1 = self.p1.as_xy()
        x2, y2 = self.p2.as_xy()
        x1_svg, y1_svg = transform.world_to_svg(x1, y1)
        x2_svg, y2_svg = transform.world_to_svg(x2, y2)

        cx_svg, cy_svg = transform.world_to_svg(c, 0.0)
        R_svg = transform.radius_to_svg(R)

        # angles of endpoints around the circle center in SVG coords:
        a1 = math.atan2(y1_svg - cy_svg, x1_svg - cx_svg)
        a2 = math.atan2(y2_svg - cy_svg, x2_svg - cx_svg)

        # Move along the *short* arc between a1 and a2
        raw_delta = (a2 - a1) % (2 * math.pi)
        large_arc_flag = 0
        sweep_flag = 1 if raw_delta <= math.pi else 0

        d = (
            f"M {x1_svg:.3f},{y1_svg:.3f} "
            f"A {R_svg:.3f},{R_svg:.3f} 0 {large_arc_flag} {sweep_flag} "
            f"{x2_svg:.3f},{y2s:.3f}"
        )

        return (
            f'<path d="{d}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'fill="none"/>'
        )


# ---------------------------------------------------------
# 2. UHP TRANSFORM + UHP SCENE + POLYGON
# ---------------------------------------------------------
@dataclass
class Transform:
    """
    Handles math→SVG mapping for the upper half-plane scene.
    Keeps aspect ratio isotropic so circles stay circles.
    """
    xmin: float
    xmax: float
    ymax: float
    width: int
    height: int
    pin_bottom: bool = True  # NEW: Pin y=0 to bottom of canvas

    scale: float = field(init=False)
    xpad: float = field(init=False)
    ypad: float = field(init=False)

    def __post_init__(self):
        span_x = self.xmax - self.xmin
        span_y = self.ymax - 0.0

        sx = self.width  / span_x
        sy = self.height / span_y
        self.scale = min(sx, sy)

        used_w = self.scale * span_x
        
        if self.pin_bottom:
            # Pin y=0 to the bottom, center horizontally
            self.xpad = (self.width - used_w) / 2.0
            # For y=0 to be at bottom (y_svg = height), we need:
            # height = ypad + (ymax - 0) * scale
            # ypad = height - ymax * scale
            self.ypad = self.height - self.ymax * self.scale
        else:
            # Original behavior: center both directions
            used_h = self.scale * span_y
            self.xpad = (self.width  - used_w) / 2.0
            self.ypad = (self.height - used_h) / 2.0

    def world_to_svg(self, x: float, y: float) -> Tuple[float, float]:
        sx = self.xpad + (x - self.xmin) * self.scale
        sy = self.ypad + (self.ymax - y) * self.scale  # flip y
        return (sx, sy)

    def radius_to_svg(self, R: float) -> float:
        return R * self.scale


@dataclass
class HyperbolicPolygon:
    """
    A closed polygonal chain of hyperbolic geodesic edges in the UHP.
    The vertices must be in boundary order (counterclockwise).
    """
    vertices: List[HyperbolicPoint]
    stroke: str = "purple"
    stroke_width: float = 2.0
    fill: str = "none"

    def translate(self, dx: float = 0.0, dy: float = 0.0) -> "HyperbolicPolygon":
        """
        Euclidean translate in the UHP:
        - interior (x,y>0) -> (x+dx, y+dy) [must stay y>0]
        - boundary (x,0)   -> (x+dx, 0)   (keep on boundary)
        - infinity         -> stays infinity
        """
        new_vertices: List[HyperbolicPoint] = []
        for v in self.vertices:
            if v.kind == "interior":
                new_x = v.x + dx
                new_y = v.y + dy
                if new_y <= 0:
                    raise ValueError(
                        f"translate(): moved interior point below boundary (y={new_y})"
                    )
                new_vertices.append(HyperbolicPoint("interior", new_x, new_y))

            elif v.kind == "boundary":
                new_vertices.append(HyperbolicPoint("boundary", v.x + dx))

            elif v.kind == "infinity":
                new_vertices.append(HyperbolicPoint.infinity())

            else:
                raise ValueError(f"Unknown point kind {v.kind}")

        return HyperbolicPolygon(
            vertices=new_vertices,
            stroke=self.stroke,
            stroke_width=self.stroke_width,
            fill=self.fill,
        )

    def scale(self, s: float) -> "HyperbolicPolygon":
        """
        Euclidean scale about (0,0) in UHP coords:
        - interior (x,y) -> (s*x, s*y)  [y must stay >0]
        - boundary (x,0) -> (s*x, 0)
        - infinity       -> infinity
        Note: scaling by s>0 is actually a hyperbolic isometry (a dilation).
        """
        if s <= 0:
            raise ValueError("scale() needs s>0")
        new_vertices: List[HyperbolicPoint] = []
        for v in self.vertices:
            if v.kind == "interior":
                new_x = s * v.x
                new_y = s * v.y
                if new_y <= 0:
                    raise ValueError(
                        f"scale(): moved interior point below boundary (y={new_y})"
                    )
                new_vertices.append(HyperbolicPoint("interior", new_x, new_y))

            elif v.kind == "boundary":
                new_vertices.append(HyperbolicPoint("boundary", s * v.x))

            elif v.kind == "infinity":
                new_vertices.append(HyperbolicPoint.infinity())

            else:
                raise ValueError(f"Unknown point kind {v.kind}")

        return HyperbolicPolygon(
            vertices=new_vertices,
            stroke=self.stroke,
            stroke_width=self.stroke_width,
            fill=self.fill,
        )

    def _edge_commands_svg(
        self,
        p_start: HyperbolicPoint,
        p_end: HyperbolicPoint,
        tf: Transform,
    ) -> str:
        """
        One edge as an SVG sub-command within a single <path>.
        Returns "L x,y" for vertical geodesics,
        or "A rx,ry 0 largeArcFlag sweepFlag x,y" for circular geodesics.
        """
        g = Geodesic(p_start, p_end)
        kind_params = g._circle_params()

        if kind_params[0] == "vertical":
            # For path continuity, we only need to give the end point.
            _, y2 = p_end.as_xy()
            x2 = kind_params[1]
            x2s, y2s = tf.world_to_svg(x2, y2)
            return f"L {x2s:.3f},{y2s:.3f}"

        # circle arc
        _, c, R = kind_params

        (x1, y1) = p_start.as_xy()
        (x2, y2) = p_end.as_xy()

        x1s, y1s = tf.world_to_svg(x1, y1)
        x2s, y2s = tf.world_to_svg(x2, y2)
        cxs, cys = tf.world_to_svg(c, 0.0)
        R_svg = tf.radius_to_svg(R)

        a1 = math.atan2(y1s - cys, x1s - cxs)
        a2 = math.atan2(y2s - cys, x2s - cxs)
        raw_delta = (a2 - a1) % (2 * math.pi)

        large_arc_flag = 0
        sweep_flag = 1 if raw_delta <= math.pi else 0

        return (
            f"A {R_svg:.3f},{R_svg:.3f} 0 {large_arc_flag} {sweep_flag} "
            f"{x2s:.3f},{y2s:.3f}"
        )

    def to_svg_path_element(self, tf: Transform) -> str:
        """
        Build ONE <path> element that walks the entire polygon boundary
        using geodesic edges as arc (`A`) or line (`L`) subcommands.
        We close with 'Z' for a proper loop.
        """
        if len(self.vertices) < 2:
            return ""

        # Move to first vertex
        x0, y0 = self.vertices[0].as_xy()
        x0s, y0s = tf.world_to_svg(x0, y0)
        cmds = [f"M {x0s:.3f},{y0s:.3f}"]

        n = len(self.vertices)
        for i in range(n):
            p_start = self.vertices[i]
            p_end   = self.vertices[(i+1) % n]
            cmds.append(self._edge_commands_svg(p_start, p_end, tf))

        cmds.append("Z")
        d_attr = " ".join(cmds)

        return (
            f'<path d="{d_attr}" stroke="{self.stroke}" '
            f'stroke-width="{self.stroke_width}" '
            f'stroke-linejoin="round" stroke-linecap="round" '
            f'fill="{self.fill}"/>'
        )


@dataclass
class UHPScene:
    """
    An SVG renderer for objects in the upper half-plane.
    Can render:
      - background / axis
      - polygons (as single joined paths)
      - standalone geodesic segments
      - horocycles
    """
    width: int = 600
    height: int = 400
    xmin: float = -5.0
    xmax: float =  5.0
    ymax: float =  5.0

    geodesics: List[Tuple[Geodesic, str, float]] = field(default_factory=list)
    polygons: List[HyperbolicPolygon] = field(default_factory=list)
    horocycles: List[Tuple[Horocycle, str, float, str]] = field(default_factory=list)
    background_fill: str = "#dddddd"

    def add_geodesic(
        self,
        p1: HyperbolicPoint,
        p2: HyperbolicPoint,
        stroke: str = "black",
        stroke_width: float = 2.0,
    ):
        g = Geodesic(p1, p2)
        self.geodesics.append((g, stroke, stroke_width))

    def add_polygon(self, poly: HyperbolicPolygon):
        self.polygons.append(poly)
    
    def add_horocycle(
        self,
        horocycle: Horocycle,
        stroke: str = "blue",
        stroke_width: float = 1.5,
        stroke_dasharray: str = "5,3",
    ):
        """Add a horocycle to the scene."""
        self.horocycles.append((horocycle, stroke, stroke_width, stroke_dasharray))

    def _tf(self) -> Transform:
        return Transform(
            xmin=self.xmin,
            xmax=self.xmax,
            ymax=self.ymax,
            width=self.width,
            height=self.height,
        )

    def to_svg(self) -> str:
        tf = self._tf()

        parts = [
            f'<svg xmlns="http://www.w3.org/2000/svg" '
            f'width="{self.width}" height="{self.height}" '
            f'viewBox="0 0 {self.width} {self.height}">'
        ]

        # background rect
        parts.append(
            f'<rect x="0" y="0" width="{self.width}" height="{self.height}" '
            f'fill="{self.background_fill}"/>'
        )

        # horizontal real axis y=0 (dashed guide)
        xL, yAxis = tf.world_to_svg(self.xmin, 0.0)
        xR, _     = tf.world_to_svg(self.xmax, 0.0)
        parts.append(
            f'<line x1="{xL:.3f}" y1="{yAxis:.3f}" x2="{xR:.3f}" y2="{yAxis:.3f}" '
            f'stroke="#888" stroke-width="1" stroke-dasharray="4,4"/>'
        )

        # horocycles
        for (horo, stroke, sw, dasharray) in self.horocycles:
            parts.append(
                horo.svg_element_uhp(
                    transform=tf,
                    xmin=self.xmin,
                    xmax=self.xmax,
                    stroke=stroke,
                    stroke_width=sw,
                    stroke_dasharray=dasharray,
                )
            )

        # polygons
        for poly in self.polygons:
            parts.append(poly.to_svg_path_element(tf))

        # standalone geodesic segments
        for (geo, stroke, sw) in self.geodesics:
            parts.append(
                geo.svg_element(
                    transform=tf,
                    stroke=stroke,
                    stroke_width=sw,
                    ymax_clip=self.ymax,
                )
            )

        parts.append("</svg>")
        return "\n".join(parts)


# ---------------------------------------------------------
# 3. HELPER: BUILD SPECIAL POLYGON FROM (N, a)
# ---------------------------------------------------------

def make_hyperbolic_polygon_from_params(
    N: int = 5,
    a: float = 0.55,
    stroke: str = "purple",
    stroke_width: float = 2.0,
    fill: str = "none",
) -> HyperbolicPolygon:
    """
    Reconstruct the polygon from your SDF definition:

    Let m = N-3.
    Bottom chain: (k*a, 1) for k=0..m.
    Top right:    (m*a, m)
    Top left:     (0,   m)

    Returned in counterclockwise boundary order.
    """
    if N < 3:
        raise ValueError("N must be >= 3.")
    m = N - 3

    verts: List[HyperbolicPoint] = []
    # bottom rows
    for k in range(m + 1):
        verts.append(HyperbolicPoint.interior(complex(k * a, 1.0)))
    # top-right corner
    verts.append(HyperbolicPoint.interior(complex(m * a, float(m))))
    # top-left corner
    verts.append(HyperbolicPoint.interior(complex(0.0, float(m))))

    return HyperbolicPolygon(
        vertices=verts,
        stroke=stroke,
        stroke_width=stroke_width,
        fill=fill,
    )


def make_UHP_scene_for_polygon(
    N: int,
    a: float,
    width: int = 600,
    height: int = 400,
    background_fill: str = "#dddddd",
) -> UHPScene:
    """
    Convenience: choose scene bounds that fit the polygon built by
    make_hyperbolic_polygon_from_params(N,a).
    """
    m = N - 3
    xmin = -0.5 * a
    xmax = m * a + 0.5 * a
    ymax = m + 1.0

    scene = UHPScene(
        width=width,
        height=height,
        xmin=xmin,
        xmax=xmax,
        ymax=ymax,
        background_fill=background_fill,
    )
    return scene


# ---------------------------------------------------------
# 4. DISK MODEL RENDERING (Extended with Horocycles)
# ---------------------------------------------------------

def cayley_from_uhp(x: Optional[float], y: Optional[float]) -> complex:
    """
    Cayley transform φ(z) = (z - i)/(z + i) sending the UHP to the unit disk.
    x,y from HyperbolicPoint.as_xy().
    infinity -> 1 on the boundary.
    """
    if x is None and y is None:
        return 1 + 0j
    z = complex(x, y)
    return (z - 1j) / (z + 1j)


def geodesic_point_at_t(geo: Geodesic, t: float) -> Tuple[float, float]:
    """
    Return a point (x,y) in UHP along the *short* geodesic segment from geo.p1 to geo.p2.
    t=0 -> p1, t=1 -> p2. Used to recover circle geometry after mapping to the disk.
    """
    kind_params = geo._circle_params()
    x1, y1 = geo.p1.as_xy()
    x2, y2 = geo.p2.as_xy()

    if kind_params[0] == "vertical":
        x0 = kind_params[1]
        y_mid = y1 + (y2 - y1) * t
        return (x0, y_mid)

    # circle case
    _, c, R = kind_params
    a1 = math.atan2(y1 - 0.0, x1 - c)
    a2 = math.atan2(y2 - 0.0, x2 - c)

    # interpolate the short way
    da = (a2 - a1) % (2 * math.pi)
    if da > math.pi:
        da -= 2 * math.pi
    amid = a1 + da * t

    xm = c + R * math.cos(amid)
    ym =      R * math.sin(amid)
    return (xm, ym)


def circle_from_3pts_svg(xa, ya, xb, yb, xc, yc):
    """
    Circumcircle in SVG pixel coords. Returns (cx, cy, Rpx).
    If degenerate, returns None.
    """
    A = xb - xa
    B = yb - ya
    C = xc - xa
    D = yc - ya
    E = A * (xa + xb) + B * (ya + yb)
    F = C * (xa + xc) + D * (ya + yc)
    G = 2 * (A * (yc - yb) - B * (xc - xb))
    if abs(G) < 1e-12:
        return None
    cx = (D * E - B * F) / G
    cy = (A * F - C * E) / G
    Rpx = math.hypot(cx - xa, cy - ya)
    return cx, cy, Rpx


def map_geodesic_edge_to_disc_arc(
    geo: Geodesic,
    disc_to_svg,
):
    """
    Take a UHP geodesic 'geo', sample it (p1, midpoint, p2),
    Cayley-transform those samples into the unit disk,
    map to SVG coords via disc_to_svg,
    and build the correct SVG segment for that edge.

    Returns a dict:
      kind == "line":  {"kind":"line","x2":...,"y2":...}
      kind == "arc" :  {"kind":"arc","x2":...,"y2":...,
                         "R":..., "large":0/1, "sweep":0/1}
    """

    # sample three UHP points along the SAME hyperbolic geodesic
    x1, y1 = geo.p1.as_xy()
    x2, y2 = geo.p2.as_xy()
    xm, ym = geodesic_point_at_t(geo, 0.5)

    # Cayley -> unit disk
    w1 = cayley_from_uhp(x1, y1)
    w2 = cayley_from_uhp(x2, y2)
    wm = cayley_from_uhp(xm, ym)

    # map to SVG coords
    x1s, y1s = disc_to_svg(w1)
    x2s, y2s = disc_to_svg(w2)
    xms, yms = disc_to_svg(wm)

    # test collinearity => straight line (diameter chord)
    area2 = (x1s * (y2s - yms) +
             x2s * (yms - y1s) +
             xms * (y1s - y2s))
    if abs(area2) < 1e-9:
        return {
            "kind": "line",
            "x2": x2s,
            "y2": y2s,
        }

    # otherwise fit circle through the 3 points in SVG coords
    circle_fit = circle_from_3pts_svg(x1s, y1s, x2s, y2s, xms, yms)
    if circle_fit is None:
        # numerically almost collinear => fallback line
        return {
            "kind": "line",
            "x2": x2s,
            "y2": y2s,
        }

    cx, cy, Rpx = circle_fit

    # figure sweep direction using angles about (cx,cy)
    a1 = math.atan2(y1s - cy, x1s - cx)
    a2 = math.atan2(y2s - cy, x2s - cx)
    dtheta = (a2 - a1) % (2 * math.pi)
    if dtheta > math.pi:
        dtheta -= 2 * math.pi

    large_arc_flag = 0           # we always take the shorter arc
    sweep_flag = 1 if dtheta >= 0 else 0

    return {
        "kind": "arc",
        "x2": x2s,
        "y2": y2s,
        "R": Rpx,
        "large": large_arc_flag,
        "sweep": sweep_flag,
    }


def map_horocycle_to_disc(
    horo: Horocycle,
    disc_to_svg,
    n_samples: int = 100,
) -> Optional[Tuple[float, float, float]]:
    """
    Map a horocycle from UHP to the Poincaré disk.
    Returns (cx_svg, cy_svg, R_svg) for the circle in SVG coordinates,
    or None if it can't be determined.
    
    For horizontal horocycles y=h, we sample points and fit a circle.
    """
    if horo.kind == "horizontal":
        # Sample points along y = h from x in a reasonable range
        # We'll use x from -10 to 10 (adjust as needed)
        x_vals = [i * 20.0 / (n_samples - 1) - 10.0 for i in range(n_samples)]
        
        # Transform to disk and then to SVG
        svg_points = []
        for x in x_vals:
            w = cayley_from_uhp(x, horo.h)
            xs, ys = disc_to_svg(w)
            svg_points.append((xs, ys))
        
        # Fit circle through first, middle, and last point
        if len(svg_points) >= 3:
            p1 = svg_points[0]
            p2 = svg_points[len(svg_points)//2]
            p3 = svg_points[-1]
            
            circle_fit = circle_from_3pts_svg(
                p1[0], p1[1],
                p2[0], p2[1],
                p3[0], p3[1]
            )
            if circle_fit:
                return circle_fit
    
    elif horo.kind == "circle":
        # Circle tangent to real axis: sample points on the circle
        angles = [i * 2 * math.pi / n_samples for i in range(n_samples)]
        svg_points = []
        
        for theta in angles:
            # Point on circle centered at (c, R) with radius R
            x = horo.c + horo.R * math.cos(theta)
            y = horo.R + horo.R * math.sin(theta)
            
            if y > 0:  # Only upper half
                w = cayley_from_uhp(x, y)
                xs, ys = disc_to_svg(w)
                svg_points.append((xs, ys))
        
        # Fit circle through evenly spaced points
        if len(svg_points) >= 3:
            idx1 = 0
            idx2 = len(svg_points) // 3
            idx3 = 2 * len(svg_points) // 3
            
            p1 = svg_points[idx1]
            p2 = svg_points[idx2]
            p3 = svg_points[idx3]
            
            circle_fit = circle_from_3pts_svg(
                p1[0], p1[1],
                p2[0], p2[1],
                p3[0], p3[1]
            )
            if circle_fit:
                return circle_fit
    
    return None


@dataclass
class DiscScene:
    """
    An SVG renderer for the Poincaré disk.
    We DON'T store "disc points"; instead, we store polygons and horocycles in UHP coords.
    On render, we Cayley-transform edges correctly (including curvature).
    """
    width: int = 500
    height: int = 500
    geodesics: List[Tuple[Geodesic, str, float]] = field(default_factory=list)  # ADD THIS LINE
    polygons: List[HyperbolicPolygon] = field(default_factory=list)
    horocycles: List[Tuple[Horocycle, str, float, str]] = field(default_factory=list)
    background_fill: str = "#dddddd"

    def add_geodesic(
        self,
        p1: HyperbolicPoint,
        p2: HyperbolicPoint,
        stroke: str = "black",
        stroke_width: float = 2.0,
    ):
        """Add a geodesic segment to the scene."""
        g = Geodesic(p1, p2)
        self.geodesics.append((g, stroke, stroke_width))
            
    def add_polygon(self, poly: HyperbolicPolygon):
        self.polygons.append(poly)
    
    def add_horocycle(
        self,
        horocycle: Horocycle,
        stroke: str = "blue",
        stroke_width: float = 1.5,
        stroke_dasharray: str = "5,3",
    ):
        """Add a horocycle to the scene."""
        self.horocycles.append((horocycle, stroke, stroke_width, stroke_dasharray))

    def _R(self):
        # radius (in px) used for drawing the unit disk
        return 0.45 * min(self.width, self.height)

    def _disc_to_svg(self, w: complex) -> Tuple[float, float]:
        cx = self.width / 2.0
        cy = self.height / 2.0
        R = self._R()
        return (cx + R * w.real, cy - R * w.imag)

    def to_svg(self) -> str:
        cx = self.width / 2.0
        cy = self.height / 2.0
        R = self._R()

        parts = [
            f'<svg xmlns="http://www.w3.org/2000/svg" '
            f'width="{self.width}" height="{self.height}" '
            f'viewBox="0 0 {self.width} {self.height}">'
        ]

        # draw the unit disk boundary
        parts.append(
            f'<circle cx="{cx:.3f}" cy="{cy:.3f}" r="{R:.3f}" '
            f'fill="{self.background_fill}" stroke="black" stroke-width="1"/>'
        )
        # draw geodesics
        for (geo, stroke, sw) in self.geodesics:
            seginfo = map_geodesic_edge_to_disc_arc(
                geo,
                disc_to_svg=self._disc_to_svg,
            )
            
            # Get start point
            x1, y1 = geo.p1.as_xy()
            w1 = cayley_from_uhp(x1, y1)
            x1s, y1s = self._disc_to_svg(w1)
            
            if seginfo["kind"] == "line":
                parts.append(
                    f'<line x1="{x1s:.3f}" y1="{y1s:.3f}" '
                    f'x2="{seginfo["x2"]:.3f}" y2="{seginfo["y2"]:.3f}" '
                    f'stroke="{stroke}" stroke-width="{sw}"/>'
                )
            else:
                d = (
                    f'M {x1s:.3f},{y1s:.3f} '
                    f'A {seginfo["R"]:.3f},{seginfo["R"]:.3f} 0 '
                    f'{seginfo["large"]} {seginfo["sweep"]} '
                    f'{seginfo["x2"]:.3f},{seginfo["y2"]:.3f}'
                )
                parts.append(
                    f'<path d="{d}" stroke="{stroke}" '
                    f'stroke-width="{sw}" fill="none"/>'
                )
        # draw horocycles
        for (horo, stroke, sw, dasharray) in self.horocycles:
            circle_params = map_horocycle_to_disc(horo, self._disc_to_svg)
            if circle_params:
                cx_h, cy_h, R_h = circle_params
                parts.append(
                    f'<circle cx="{cx_h:.3f}" cy="{cy_h:.3f}" r="{R_h:.3f}" '
                    f'stroke="{stroke}" stroke-width="{sw}" '
                    f'stroke-dasharray="{dasharray}" fill="none"/>'
                )

        # draw each polygon
        for poly in self.polygons:
            verts = poly.vertices
            n = len(verts)
            if n < 2:
                continue

            # list of geodesics around the polygon boundary in UHP
            edges = [Geodesic(verts[i], verts[(i+1)%n]) for i in range(n)]

            # start path at first vertex (mapped to disk)
            x0, y0 = verts[0].as_xy()
            w0 = cayley_from_uhp(x0, y0)
            x0s, y0s = self._disc_to_svg(w0)
            seg_cmds = [f"M {x0s:.3f},{y0s:.3f}"]

            # add each boundary edge as correct arc or straight segment
            for geo in edges:
                seginfo = map_geodesic_edge_to_disc_arc(
                    geo,
                    disc_to_svg=self._disc_to_svg,
                )

                if seginfo["kind"] == "line":
                    seg_cmds.append(
                        f"L {seginfo['x2']:.3f},{seginfo['y2']:.3f}"
                    )
                else:
                    seg_cmds.append(
                        "A {R:.3f},{R:.3f} 0 {large} {sweep} {x2:.3f},{y2:.3f}"
                        .format(
                            R=seginfo["R"],
                            large=seginfo["large"],
                            sweep=seginfo["sweep"],
                            x2=seginfo["x2"],
                            y2=seginfo["y2"],
                        )
                    )

            seg_cmds.append("Z")
            d_attr = " ".join(seg_cmds)

            parts.append(
                f'<path d="{d_attr}" fill="{poly.fill}" '
                f'stroke="{poly.stroke}" stroke-width="{poly.stroke_width}" '
                f'stroke-linejoin="round" stroke-linecap="round" />'
            )

        parts.append("</svg>")
        return "\n".join(parts)

In [ ]:
from IPython.display import SVG, display

import os

output_dir = "svg_images"
os.makedirs(output_dir, exist_ok=True)


width = 1000
height = width

uhp_scene = UHPScene(width=width, height=height, background_fill="#ffffff")
disc_scene = DiscScene(width=width, height=height, background_fill="#ffffff")

N = 6
a = 0.5
n_layers = 5
stroke_width = 1.0

XSHIFT = 0.0  # global shift along the real axis

base_poly = make_hyperbolic_polygon_from_params(
    N, a,
    stroke="purple",
    stroke_width=stroke_width,
    # fill="green"
)

# add the big, original polygon at full size
uhp_scene.add_polygon(base_poly)
disc_scene.add_polygon(base_poly)

scale_factor = N - 3  # == 3 when N=6
for layer in range(n_layers):
    s = scale_factor ** (-(layer + 1))
    num_copies = 100*scale_factor ** (layer + 1)

    # choose stroke width for THIS layer (independent of the base polygon)
    layer_stroke_width = stroke_width #* s #math.sqrt(s)

    for k in range(num_copies):
        dx = (a / (scale_factor ** (layer))) * k

        poly_scaled_shifted_right = base_poly.scale(s).translate(dx=dx + XSHIFT)
        poly_scaled_shifted_left = base_poly.scale(s).translate(dx=-dx + XSHIFT)

        # now override style on THIS copy only
        poly_scaled_shifted_right.stroke = "purple"
        poly_scaled_shifted_right.stroke_width = layer_stroke_width
        poly_scaled_shifted_right.fill = "none"

        poly_scaled_shifted_left.stroke = "purple"
        poly_scaled_shifted_left.stroke_width = layer_stroke_width
        poly_scaled_shifted_left.fill = "none"

        uhp_scene.add_polygon(poly_scaled_shifted_right)
        disc_scene.add_polygon(poly_scaled_shifted_right)
        
        uhp_scene.add_polygon(poly_scaled_shifted_left)
        disc_scene.add_polygon(poly_scaled_shifted_left)


for layer in range(n_layers):
    s = scale_factor ** (layer)
    num_copies = 10*scale_factor ** (layer + 1)

    # choose stroke width for THIS layer (independent of the base polygon)
    layer_stroke_width = stroke_width  #math.sqrt(s)

    for k in range(num_copies):
        dx = (a * (scale_factor ** (layer+1))) * k

        poly_scaled_shifted_right = base_poly.scale(s).translate(dx=dx)
        poly_scaled_shifted_left = base_poly.scale(s).translate(dx=-dx)

        # now override style on THIS copy only
        poly_scaled_shifted_right.stroke = "purple"
        poly_scaled_shifted_right.stroke_width = layer_stroke_width
        poly_scaled_shifted_right.fill = "none"

        poly_scaled_shifted_left.stroke = "purple"
        poly_scaled_shifted_left.stroke_width = layer_stroke_width
        poly_scaled_shifted_left.fill = "none"

        uhp_scene.add_polygon(poly_scaled_shifted_right)
        disc_scene.add_polygon(poly_scaled_shifted_right)
        
        uhp_scene.add_polygon(poly_scaled_shifted_left)
        disc_scene.add_polygon(poly_scaled_shifted_left)

# uhp_scene.polygons[1].fill = "lightgreen"
# uhp_scene.polygons[10].fill = "darkgreen"
display(SVG(uhp_scene.to_svg()))
display(SVG(disc_scene.to_svg()))
# write disc scene to file 
filename = os.path.join(output_dir, f"hyperbolic_polygon_disc_N{N}_a{a}_layers{n_layers}.svg")
with open(filename, "w") as f:
    f.write(disc_scene.to_svg())
# write uhp scene to file
filename = os.path.join(output_dir, f"hyperbolic_polygon_uhp_N{N}_a{a}_layers{n_layers}.svg")
with open(filename, "w") as f:
    f.write(uhp_scene.to_svg())


In [ ]:
# Cell: Demo of horizontal horocycles with vertical segments in all layers
# This creates SVGs showing horocycles at y = base^k and vertical segments

from IPython.display import SVG, display

import os

output_dir = "svg_images"
os.makedirs(output_dir, exist_ok=True)


nlayers = 7
base = 2  # Integer base
a = 1  # base spacing for vertical segments
x_shift = 0.0  # shift along the real axis

width = 1000
height = width

# Create horizontal horocycles at y = base^k
k_values = range(-nlayers, nlayers+1)
heights = [base**k for k in k_values]

# ============================================
# 1. UHP Scene with horocycles and vertical segments
# ============================================
scene_uhp = UHPScene(
    width=width,
    height=height,
    xmin=-100.0,
    xmax=50.0,
    ymax=35.0,
    background_fill="#ffffff"
)

# Add all the horocycles
colors = [
    "#ff0000",  # red
    "#ff6600",  # orange-red
    "#ff9900",  # orange
    "#ffcc00",  # yellow-orange
    "#cccc00",  # yellow
    "#00cc00",  # green
    "#0099cc",  # cyan
    "#0066cc",  # blue
    "#3333cc",  # dark blue
    "#6600cc",  # purple
    "#9900cc",  # violet
]

color = "#000000"  # a nice b

for i, (k, h) in enumerate(zip(k_values, heights)):
    color = colors[i % len(colors)]
    scene_uhp.add_horocycle(
        Horocycle.horizontal(h),
        stroke=color,
        stroke_width=2.0,
        stroke_dasharray="none"
    )

# Add vertical segments in ALL layers
# Layer k connects y = base^k to y = base^(k+1)
# Segments are placed at x = n * (base^k * a) + x_shift for integer n
for k in k_values[:-1]:  # Don't need the last one since there's no layer above it
    y_bottom = base**k
    y_top = base**(k+1)
    
    # Skip if this layer is outside our viewing window
    # if y_bottom > scene_uhp.ymax:
    #     continue
    
    # Spacing for this layer: base^k * a
    spacing = (base**k) * a
    
    # Determine how many segments to add based on scene bounds and spacing
    if spacing > 0:
        max_n = int(math.ceil((scene_uhp.xmax - x_shift) / spacing)) + 1
        min_n = int(math.floor((scene_uhp.xmin - x_shift) / spacing)) - 1
        
        for n in range(min_n, max_n + 1):
            x_coord = n * spacing + x_shift
            
            # Only add if within scene bounds
            if scene_uhp.xmin <= x_coord <= scene_uhp.xmax:
                scene_uhp.add_geodesic(
                    HyperbolicPoint.interior(complex(x_coord, y_bottom)),
                    HyperbolicPoint.interior(complex(x_coord, y_top)),
                    stroke="black",
                    stroke_width=1.0
                )

# Save UHP scene
svg_uhp = scene_uhp.to_svg()
display(SVG(svg_uhp))
filename = os.path.join(output_dir, f"horocycle_b_ary_uhp_b{base}_a{a}_layers{nlayers}.svg")
with open(filename, "w") as f:
    f.write(svg_uhp)

print(f"✓ Created UHP scene with horocycles and vertical segments in all layers")

# ============================================
# 2. Disk Scene with horocycles and vertical segments
# ============================================
scene_disc = DiscScene(
    width=width,
    height=height,
    background_fill="#ffffff"
)

# Add the same horocycles
for i, (k, h) in enumerate(zip(k_values, heights)):
    color = colors[i % len(colors)]
    scene_disc.add_horocycle(
        Horocycle.horizontal(h),
        stroke=color,
        stroke_width=2.0,
        stroke_dasharray="none"
    )

# Add vertical segments in ALL layers for the disk scene
for k in k_values[:-1]:
    y_bottom = base**k
    y_top = base**(k+1)
    
    # Skip if this layer is outside reasonable bounds
    # if y_bottom > scene_uhp.ymax:
    #     continue
    
    # Spacing for this layer: base^k * a
    spacing = (base**k) * a
    
    # Determine how many segments to add
    if spacing > 0:
        max_n = int(math.ceil((scene_uhp.xmax - x_shift) / spacing)) + 1
        min_n = int(math.floor((scene_uhp.xmin - x_shift) / spacing)) - 1
        
        for n in range(min_n, max_n + 1):
            x_coord = n * spacing + x_shift
            
            # Only add if within scene bounds
            if scene_uhp.xmin <= x_coord <= scene_uhp.xmax:
                scene_disc.add_geodesic(
                    HyperbolicPoint.interior(complex(x_coord, y_bottom)),
                    HyperbolicPoint.interior(complex(x_coord, y_top)),
                    stroke="black",
                    stroke_width=1.0
                )

# Save Disk scene
svg_disc = scene_disc.to_svg()
filename = os.path.join(output_dir, f"horocycle_b_ary_disc_b{base}_a{a}_layers{nlayers}.svg")
with open(filename, "w") as f:
    f.write(svg_disc)
display(SVG(svg_disc))
print("✓ Created Disk scene with horocycles and vertical segments in all layers")

In [ ]:
from IPython.display import SVG, display

import os

output_dir = "svg_images"
os.makedirs(output_dir, exist_ok=True)


width = 1000
height = width

uhp_scene = UHPScene(width=width, height=height, background_fill="#ffffff")
disc_scene = DiscScene(width=width, height=height, background_fill="#ffffff")

N = 6
a = 1.0
n_layers = 10
stroke_width = 1.0
base_poly = make_hyperbolic_polygon_from_params(
    N, a,
    stroke="#457b9d", # nice winter color
    stroke_width=stroke_width,
    fill="#a8dadc"  # nice winter color
)

base_poly = base_poly.scale(100.0)
base_poly_width = base_poly.vertices[-2].as_xy()[0] - base_poly.vertices[0].as_xy()[0]
base_poly = base_poly.translate(-2*base_poly_width)
# uhp_scene.add_polygon(base_poly)
# disc_scene.add_polygon(base_poly)

# colors = [ "#ffd6a5", "#fdffb6", "#caffbf"]  # pastel colors
# some stylish red, green, and blue
colors = ["#ff595e", "#1982c4", "#8ac926"]
# a more vibrant version of red green and blue
# colors = ["#ff2e63", "#08d985", "#4385ff"]
# an eye friendly version of red green and blue
# colors = ["#ff6b6b", "#4ecdc4", "#1a5353"]

# define a map that checks if a color is in colors and returns the index inside colors
def color_index(color):
    if color in colors:
        return colors.index(color)
    else:
        return -1

base_poly_right = base_poly.translate(base_poly_width)
base_poly_right.fill = colors[0]

uhp_scene.add_polygon(base_poly_right)
disc_scene.add_polygon(base_poly_right)


def generate_children(parent, colors, parent_color_index = 0, plus_one=False):
    # Generate child polygons by scaling down and placing them in the bottom region
    # If colors provided, cycle through them for different children
    offset = parent.vertices[0].as_xy()[0]  # x-coordinate of first vertex
    parent = parent.translate(-offset)
    N = len(parent.vertices)
    children = []
    scale_factor = N - 3  # == 3 when N=6
    s = 1 / scale_factor
    num_copies = N-3
    colors = [c for c in colors if c != parent.fill]

    for k in range(num_copies):
        # dx is the width of parent, i.e. the difrence between the last and second to last vertex x-coordinates
        dx = (parent.vertices[-2].as_xy()[0] - parent.vertices[-1].as_xy()[0])*k/(N-3)
        poly_scaled_shifted = parent.scale(s).translate(dx=dx)
        poly_scaled_shifted.fill = colors[(parent_color_index + k) % len(colors)]
        children.append(poly_scaled_shifted.translate(offset))
    if plus_one:
        # add one extra child at the end
        dx = (parent.vertices[-2].as_xy()[0] - parent.vertices[-1].as_xy()[0])*(num_copies)/(N-3)
        poly_scaled_shifted = parent.scale(s).translate(dx=dx)
        poly_scaled_shifted.fill = parent.fill
        children.append(poly_scaled_shifted.translate(offset))
    return children

def generate_next_layer(layer, colors=None):
    next_layer = []
    for parent in layer[1:-1]: #skip first and last polygon
        color_index_parent = colors.index(parent.fill)
        children = generate_children(parent, colors=colors, parent_color_index=color_index_parent)
        next_layer.extend(children)
    next_layer.extend(generate_children(layer[-1],colors=colors, plus_one=True)) #first polygon
    return next_layer
    
layer0 = [base_poly, base_poly_right]
for poly in layer0:
    uhp_scene.add_polygon(poly)
    disc_scene.add_polygon(poly)

layers = [layer0]
layer = layer0
for i in range(1, n_layers):
    next_layer = generate_next_layer(layer, colors=colors)
    for poly in next_layer:
        uhp_scene.add_polygon(poly)
        disc_scene.add_polygon(poly)
    layer = next_layer

svg_uhp = uhp_scene.to_svg()
svg_disc = disc_scene.to_svg()
display(SVG(svg_uhp))
display(SVG(svg_disc))

filename = os.path.join(output_dir, f"layered_polygon_uhp_N{N}_a{a}_layers{n_layers}.svg")
with open(filename, "w") as f:
    f.write(svg_uhp)
filename = os.path.join(output_dir, f"layered_polygon_disc_N{N}_a{a}_layers{n_layers}.svg")
with open(filename, "w") as f:
    f.write(svg_disc)

# for layer in range(n_layers):
#     s = scale_factor ** (-(layer + 1))
#     num_copies = 100*scale_factor ** (layer + 1)

#     # choose stroke width for THIS layer (independent of the base polygon)
#     layer_stroke_width = stroke_width #* s #math.sqrt(s)

#     for k in range(num_copies):
#         dx = (a / (scale_factor ** (layer))) * k

#         poly_scaled_shifted_right = base_poly.scale(s).translate(dx=dx)

In [ ]:
# Simple side-by-side view of the base polygon in UHP and disk
from IPython.display import SVG, display

import os

output_dir = "svg_images"
os.makedirs(output_dir, exist_ok=True)


N = 5
a = 2.0
width = 400
height = 400
gap = 20

uhp_scene = UHPScene(width=width, height=height, background_fill="#ffffff")
disc_scene = DiscScene(width=width, height=height, background_fill="#ffffff")

poly = make_hyperbolic_polygon_from_params(
    N, a,
    stroke="#2b2d42",
    stroke_width=2.0,
    fill="#edf2f4",
)

uhp_scene.add_polygon(poly)
disc_scene.add_polygon(poly)

def _inner_svg(svg_str):
    start = svg_str.find(">") + 1
    end = svg_str.rfind("</svg>")
    return svg_str[start:end]

uhp_inner = _inner_svg(uhp_scene.to_svg())
disc_inner = _inner_svg(disc_scene.to_svg())

combined_width = width * 2 + gap
combined_height = height

combined_svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{combined_width}" height="{combined_height}" viewBox="0 0 {combined_width} {combined_height}">
  <rect x="0" y="0" width="{combined_width}" height="{combined_height}" fill="white"/>
  <g transform="translate(0,0)">
    {uhp_inner}
  </g>
  <g transform="translate({width + gap},0)">
    {disc_inner}
  </g>
  <text x="{width/2}" y="20" text-anchor="middle" font-size="14" font-family="serif">UHP</text>
  <text x="{width + gap + width/2}" y="20" text-anchor="middle" font-size="14" font-family="serif">Disk</text>
</svg>'''

display(SVG(combined_svg))

filename = os.path.join(output_dir, f"polygon_side_by_side_N{N}_a{a}.svg")
# with open(filename, "w") as f:
#     f.write(combined_svg)


# Hyperbolic Widths

In [ ]:
import math
import drawsvg as draw
from hyperbolic.poincare import Point, Line
from IPython.display import display

In [ ]:
# === Core functions ===

def uhp_to_disc(x: float, y: float) -> Point:
    """Convert UHP point (x, y) to Poincaré disc Point.
    
    Uses the same Möbius transform as the hyperbolic library:
    w = (iz + 1) / (z + i)  =  i * (z - i) / (z + i)
    """
    z = complex(x, y)
    # Library convention: rotated by i compared to standard Cayley
    w = (1j * z + 1) / (z + 1j)
    return Point(w.real, w.imag)


def make_polygon_vertices_uhp(N: int, a: float):
    """
    Create polygon vertices in UHP coordinates.
    
    From the notebook:
    - m = N - 3
    - Bottom chain: (k*a, 1) for k = 0..m
    - Top right: (m*a, m)
    - Top left: (0, m)
    """
    m = N - 3
    vertices = []
    for k in range(m + 1):
        vertices.append((k * a, 1.0))
    vertices.append((m * a, float(m)))
    vertices.append((0.0, float(m)))
    return vertices


def scale_uhp(verts, s):
    """Scale vertices in UHP (hyperbolic isometry)."""
    return [(x * s, y * s) for x, y in verts]


def translate_uhp(verts, dx, dy=0):
    """Translate vertices in UHP."""
    return [(x + dx, y + dy) for x, y in verts]


def polygon_width(verts):
    """Get the width of a polygon (x extent)."""
    xs = [x for x, y in verts]
    return max(xs) - min(xs)


def draw_hyperbolic(drawing, obj, scale=350, **kwargs):
    """Draw a hyperbolic object scaled to pixel coords."""
    drawables = obj.to_drawables(**kwargs)
    g = draw.Group(transform=f'scale({scale}, {-scale})')
    for d in drawables:
        g.append(d)
    drawing.append(g)


def draw_polygon_hwidth(drawing, uhp_verts, fill_color, stroke_color, hwidth, scale):
    """Draw a polygon with hyperbolic edge width."""
    vertices = [uhp_to_disc(x, y) for x, y in uhp_verts]
    n = len(vertices)
    
    try:
        edges = [Line.from_points(vertices[i].x, vertices[i].y,
                                  vertices[(i+1)%n].x, vertices[(i+1)%n].y,
                                  segment=True) for i in range(n)]
        poly = Polygon.from_edges(edges, join=True)
        
        if fill_color and fill_color != 'none':
            draw_hyperbolic(drawing, poly, scale=scale, fill=fill_color, stroke='none')
        
        for edge in edges:
            draw_hyperbolic(drawing, edge, scale=scale, hwidth=hwidth, fill=stroke_color, stroke='none')
    except:
        pass  # Skip problematic polygons

In [ ]:
# === Rightward Tiling with Shift ===
# Duplicate polygons only to the right, then shift the whole scene left to center.

N = 5
a = 0.5
scale = 320
hwidth = 0.052

n_copies = 50      # Number of base polygons to the RIGHT only
shift_amount = n_copies // 8  # Shift scene left by this many base widths
n_layers = 8       # Subdivision layers
n_up_layers = 3    # Upscaled background layers

colors = ['#ff595e', '#1982c4', '#8ac926']
stroke_color = '#1a1a2e'

# === Setup ===
size = 750
d = draw.Drawing(size, size, origin='center')
d.append(draw.Circle(0, 0, scale, fill='#f8f9fa', stroke='black', stroke_width=2))

clip = draw.ClipPath(id='disc-clip-right')
clip.append(draw.Circle(0, 0, scale))
d.append(clip)
g_clipped = draw.Group(clip_path='url(#disc-clip-right)')

# === Build tiling ===
base_uhp = make_polygon_vertices_uhp(N, a)
m = N - 3
base_width = polygon_width(base_uhp)

# Global shift to center the view
global_shift = -shift_amount * base_width

all_polygons = []  # (uhp_verts, color_index, layer)

# 1. Base polygons - only to the RIGHT (h = 0 to n_copies)
for h in range(n_copies + 1):
    shifted = translate_uhp(base_uhp, h * base_width + global_shift)
    all_polygons.append((shifted, 0, 0))

# 2. Scaled-down layers (smaller polygons towards boundary)
for layer in range(1, n_layers + 1):
    s = m ** (-layer)
    num_per_base = m ** layer

    for h in range(n_copies + 1):
        for k in range(num_per_base):
            dx = (a / (m ** (layer - 1))) * k
            scaled_shifted = translate_uhp(scale_uhp(base_uhp, s),
                                           h * base_width + dx + global_shift)
            all_polygons.append((scaled_shifted, layer % len(colors), layer))

# 3. Upscaled layers (larger background polygons)
for layer in range(1, n_up_layers + 1):
    s = m ** layer
    scaled_base = scale_uhp(base_uhp, s)
    scaled_width = polygon_width(scaled_base)

    # Fewer copies needed for larger polygons
    n_large = max(1, n_copies // (m ** layer) + 2)
    for h in range(-1, n_large):
        shifted = translate_uhp(scaled_base, h * scaled_width + global_shift)
        all_polygons.append((shifted, (layer + 1) % len(colors), -layer))

# Sort by layer (draw background first)
all_polygons.sort(key=lambda x: x[2])

print(f"Total polygons: {len(all_polygons)}")
print(f"Global shift: {global_shift} (in UHP x-coordinates)")

# Draw all
for uhp_verts, color_idx, layer in all_polygons:
    layer_hwidth = hwidth * (0.7 ** abs(layer)) if layer != 0 else hwidth
    draw_polygon_hwidth(g_clipped, uhp_verts,
                        colors[color_idx], stroke_color,
                        layer_hwidth, scale)

d.append(g_clipped)
d.append(draw.Circle(0, 0, scale, fill='none', stroke='black', stroke_width=2))

display(d)

d.save_svg('tiling_rightward.svg')
print("Saved to tiling_rightward.svg")

In [ ]:
# Self-contained Jupyter cell:
# Top-down tiling:
#   - Start from ONE very large polygon in the UHP (scaled up)
#   - Recursively generate children (your 3-color algorithm)
#   - Stop by max_depth and/or minimum UHP width (to avoid infinite growth)
# Rendering:
#   - For each polygon: UHP -> DISK (rounded like your "hard-coded works" approach)
#   - Build polygon EDGEWISE (Line segments) + Polygon(edges, join=True)
#   - Draw with hyperbolic thick border (hwidth) + colored fill
#   - Render in DISK and also in UHP via transform=disk_to_half()
#
# Requires: pip install hyperbolic drawsvg

import drawsvg as draw
from hyperbolic import euclid
from hyperbolic.poincare import Point, Line, Polygon
from hyperbolic.poincare import Transform

# -----------------------------
# Base polygon in UHP
# -----------------------------
def make_polygon_vertices_uhp(N: int, a: float):
    m = N - 3
    verts = []
    for k in range(m + 1):
        verts.append((k * a, 1.0))
    verts.append((m * a, float(m)))
    verts.append((0.0, float(m)))
    return verts

def translate_uhp(verts, dx=0.0, dy=0.0):
    return [(x + dx, y + dy) for (x, y) in verts]

def scale_uhp(verts, s):
    # scale about origin in UHP coords (simple + consistent with your earlier algorithms)
    return [(s * x, s * y) for (x, y) in verts]

def polygon_width(verts):
    xs = [x for x, y in verts]
    return max(xs) - min(xs)

# -----------------------------
# 3-color child algorithm (extracted + adapted for top-down recursion)
# -----------------------------
def generate_children(uhp_verts, parent_ci, colors, plus_one=False):
    """
    Geometry:
      - normalize so first vertex x=0
      - scale by s = 1/(N-3)
      - create m=N-3 children placed along bottom chain
      - translate back
    Coloring:
      - children only use the two colors != parent's color
      - cycle alternately, phase depends on parent_ci and k
      - optional plus_one: append one extra child with parent's color
    """
    offset = uhp_verts[0][0]
    verts0 = translate_uhp(uhp_verts, dx=-offset)

    Np = len(verts0)
    m = Np - 3
    s = 1.0 / m
    num = m

    # two allowed colors (indices) excluding parent_ci
    allowed = [i for i in range(3) if i != parent_ci]  # length 2

    # width measure used in your snippet: x[-2] - x[-1]
    dx_total = verts0[-2][0] - verts0[-1][0]

    children = []
    for k in range(num):
        dx = dx_total * k / m
        child_verts = translate_uhp(scale_uhp(verts0, s), dx=dx)
        child_ci = allowed[(parent_ci + k) % 2]
        children.append((translate_uhp(child_verts, dx=offset), child_ci))

    if plus_one:
        dx = dx_total * num / m
        child_verts = translate_uhp(scale_uhp(verts0, s), dx=dx)
        children.append((translate_uhp(child_verts, dx=offset), parent_ci))

    return children

# -----------------------------
# Parameters
# -----------------------------
N = 5
a = 0.5

# "Very large polygon high up":
# choose L so scale = (N-3)^L is big
L = 5  # try 3,4,5
m = N - 3
S0 = (m ** L)

# recursion / stopping
max_depth = 6
min_width_uhp = 0.5 * a      # stop subdividing when polygon width gets small

# drawing
hwidth_base = 0.082
stroke_color = "#1a1a2e"
# stroke_color = "rgba(26, 26, 46, 0.1)"  # translucent black
colors = ["#ff595e", "#1982c4", "#8ac926"]
ROUND_DECIMALS = 10

# Optional: shift to center horizontally in the view
# (We center the big polygon around x=0 by shifting half its width left)
base = make_polygon_vertices_uhp(N, a)
base_big = scale_uhp(base, S0)
w0 = polygon_width(base_big)
base_big = translate_uhp(base_big, dx=-0.5 * w0)

# -----------------------------
# Build tiling (top-down BFS)
# -----------------------------
# Each item: (verts_uhp, color_index, depth)
queue = [(base_big, 0, 0)]  # start with one large polygon, color 0
polys = []                  # collected polygons to render
all_uhp_xy = []

while queue:
    verts, ci, depth = queue.pop(0)  # BFS; use pop() for DFS
    polys.append((verts, ci, depth))
    all_uhp_xy.extend(verts)

    # stopping criteria
    if depth >= max_depth:
        continue
    if polygon_width(verts) <= min_width_uhp:
        continue

    # generate children
    children = generate_children(verts, parent_ci=ci, colors=colors, plus_one=False)
    for cverts, cci in children:
        queue.append((cverts, cci, depth + 1))

print("Polygons generated:", len(polys))
print("Max depth used:", max(d for _, _, d in polys))

# Per-depth hwidth scaling (similar to your layer scaling)
def depth_hwidth(depth):
    return hwidth_base * (0.7 ** depth) if depth != 0 else hwidth_base

# -----------------------------
# Convert each polygon to DISK (rounded) and build EDGEWISE disk polygons
# -----------------------------
T_hd = Transform.half_to_disk()
T_dh = Transform.disk_to_half()

polys_disk = []   # (depth, poly_disk, fill_color)
all_disk_xy = []

for uhp_verts, ci, depth in polys:
    disk_pts = []
    for (x, y) in uhp_verts:
        xd, yd = T_hd.apply_to_tuple((x, y))
        xd = float(f"{xd:.{ROUND_DECIMALS}f}")
        yd = float(f"{yd:.{ROUND_DECIMALS}f}")
        disk_pts.append(Point(xd, yd))
        all_disk_xy.append((xd, yd))

    edges = [
        Line.from_points(*disk_pts[j], *disk_pts[(j + 1) % len(disk_pts)], segment=True)
        for j in range(len(disk_pts))
    ]
    poly_d = Polygon(edges, join=True)
    polys_disk.append((depth, poly_d, colors[ci]))

# Draw background first (small depth), finer later
polys_disk.sort(key=lambda t: t[0])

# -----------------------------
# Render in DISK (clipped to unit circle)
# -----------------------------
xs = [x for x, y in all_disk_xy] + [-1.0, 1.0]
ys = [y for x, y in all_disk_xy] + [-1.0, 1.0]
xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)
pad = 0.08
disk_viewbox = f"{xmin-pad} {ymin-pad} {(xmax-xmin)+2*pad} {(ymax-ymin)+2*pad}"

disk = draw.Drawing(1000, 900, origin="center", viewBox=disk_viewbox)
disk.append(draw.Circle(0, 0, 1, fill="#f8f9fa", stroke="black", stroke_width=0.01))

clip = draw.ClipPath(id="disc-clip")
clip.append(draw.Circle(0, 0, 1))
disk.append(clip)
gD = draw.Group(clip_path="url(#disc-clip)")

# for depth, poly, fillc in polys_disk:
#     lw = depth_hwidth(depth)
#     # hyperbolic thick border band
#     gD.draw(poly, hwidth=lw, fill=stroke_color, stroke="none", opacity=1.0)
#     # interior fill
#     gD.draw(poly, fill=fillc, stroke="none", opacity=0.80)

# Pass 1: all fills (back-to-front as you already sorted)
for depth, poly, fillc in polys_disk:
    gD.draw(poly, fill=fillc, stroke="none", opacity=0.80)

# Pass 2: all outlines on top
for depth, poly, fillc in polys_disk:
    lw = depth_hwidth(depth)
    gD.draw(poly, hwidth=lw, fill=stroke_color, stroke="none", opacity=1.0)


disk.append(gD)
disk.append(draw.Circle(0, 0, 1, fill="none", stroke="black", stroke_width=0.01))

# -----------------------------
# Render in UHP via disk_to_half()
# -----------------------------
uxs = [x for x, y in all_uhp_xy]
uys = [y for x, y in all_uhp_xy]
uxmin, uxmax = min(uxs), max(uxs)
uymax = max(uys)

padx = 0.08 * (uxmax - uxmin + 1e-9) + 1.0
pady = 0.10 * (uymax + 1e-9) + 1.0
uhp_viewbox = f"{uxmin-padx} {-0.35} {(uxmax-uxmin)+2*padx} {uymax+pady+0.35}"

uhp = draw.Drawing(1400, 650, origin="center", viewBox=uhp_viewbox)
uhp.append(draw.Line(uxmin-padx, 0, uxmax+padx, 0, stroke="black", stroke_width=0.02))

gH = draw.Group()

# Pass 1: all fills
for depth, poly, fillc in polys_disk:
    gH.draw(poly, fill=fillc, stroke="none", opacity=0.80, transform=T_dh)

# Pass 2: all outlines on top
for depth, poly, fillc in polys_disk:
    lw = depth_hwidth(depth)
    gH.draw(poly, hwidth=lw, fill=stroke_color, stroke="none", opacity=1.0, transform=T_dh)

uhp.append(gH)

# -----------------------------
# Display + save
# -----------------------------
display(disk)
display(uhp)

disk.save_svg("topdown_tiling_disk.svg")
uhp.save_svg("topdown_tiling_uhp.svg")
print("Wrote: topdown_tiling_disk.svg, topdown_tiling_uhp.svg")


In [ ]:
# Geodesic staircase at multiple heights: 2^L down to 2^(-L)
# Number of steps doubles for each layer going down

import drawsvg as draw
from hyperbolic.poincare import Point, Line, Transform
from IPython.display import display

# === Parameters ===
L = 5                   # Max scale: heights from 2^L down to 2^(-L)
a = 0.55                # Base step size
k = 1                   # Multiplier: step size = k * a * height
OFFSET = -50            # Horizontal offset
N_STEPS_BASE = 10        # Number of segments at top level (doubles each layer down)

# Drawing parameters
size = 700
scale = 320
hwidth = 0.015          # Hyperbolic width for all lines

# === UHP to Disk transform ===
T_hd = Transform.half_to_disk()

def uhp_to_disk_point(x, y):
    """Convert UHP (x, y) to Poincaré disk Point."""
    xd, yd = T_hd.apply_to_tuple((x, y))
    return Point(xd, yd)

def uhp_boundary_to_disk(x):
    """Convert UHP boundary point (x, 0) to disk boundary point."""
    xd, yd = T_hd.apply_to_tuple((x, 0.001))
    r = (xd**2 + yd**2)**0.5
    return Point(xd/r, yd/r)

# === Create drawing ===
d = draw.Drawing(size, size, origin='center')

# Unit disk background
d.append(draw.Circle(0, 0, scale, fill='#f8f9fa', stroke='black', stroke_width=2))

g = draw.Group(transform=f'scale({scale}, {-scale})')

# Draw geodesic from 0 to infinity (vertical diameter) - reference
p_zero = Point(0, -1)
p_inf = Point(0, 1)
line_inf = Line.from_points(p_zero.x, p_zero.y, p_inf.x, p_inf.y, segment=True)
g.draw(line_inf, hwidth=hwidth, fill='lightgray', stroke='none')

# Colors for different levels
level_colors = ['#e63946', '#f4a261', '#2a9d8f', '#264653', '#9b5de5', '#00bbf9', '#00f5d4', '#fee440', '#f15bb5']

# Loop through levels from L down to -L
for level in range(L, -L-1, -1):
    height = 2 ** level
    step = k * a * height  # Scale step size with height and k
    
    # Number of steps doubles for each layer going down from L
    n_steps = N_STEPS_BASE * (2 ** (L - level))
    
    # Points at this level
    uhp_points = [(OFFSET + j * step, height) for j in range(n_steps + 1)]
    disk_points = [uhp_to_disk_point(x, y) for x, y in uhp_points]
    
    color_idx = (L - level) % len(level_colors)
    color = level_colors[color_idx]
    
    # Draw vertical rays from each vertex going down to boundary
    for i, (x, y) in enumerate(uhp_points):
        p_top = disk_points[i]
        p_bottom = uhp_boundary_to_disk(x)
        ray = Line.from_points(p_top.x, p_top.y, p_bottom.x, p_bottom.y, segment=True)
        g.draw(ray, hwidth=hwidth, fill=color, stroke='none')
    
    # Draw the horizontal geodesic segments at this level
    for i in range(n_steps):
        p1 = disk_points[i]
        p2 = disk_points[i + 1]
        line = Line.from_points(p1.x, p1.y, p2.x, p2.y, segment=True)
        g.draw(line, hwidth=hwidth, fill='#1a1a2e', stroke='none')

d.append(g)

# # Mark 0 and infinity on the boundary
# d.append(draw.Circle(0, scale, 10, fill='black'))
# d.append(draw.Circle(0, -scale, 10, fill='white', stroke='black', stroke_width=2))

# # Labels
# d.append(draw.Text('0', 14, 15, scale, fill='black', font_family='serif'))
# d.append(draw.Text('∞', 14, 15, -scale, fill='black', font_family='serif'))

# # Redraw boundary on top
# d.append(draw.Circle(0, 0, scale, fill='none', stroke='black', stroke_width=2))

display(d)
print(f"Levels: 2^{L} = {2**L} down to 2^{-L} = {2**(-L)}")
print(f"Step size = k * a * height = {k} * {a} * height")
print(f"Steps at top level: {N_STEPS_BASE}, doubles each layer down")